In [1]:
import sys
from pathlib import Path

CWD = Path(__name__).resolve().parent
sys.path.append(CWD)

DATASET_FILE = CWD / "result.json"
# OUT_CHECKPOINT_FILE = CWD / "leanrag_checkpoint.json"
# USE_CHECKPOINT_AS_CACHE = True # prefer data in the checkpoint file over re-computing?

secrets = CWD / "secrets.env"

if not secrets.is_file():
    raise ValueError(f"secrets file at '{secrets}' does not exist")

from dotenv import load_dotenv
load_dotenv(secrets)

# (START WITH G0 IN MEMGRAPH) ...

True

In [2]:
# TESTING FUNCTIONS
import utils.mg_driver as mg_driver
await mg_driver.init()

layer = 0
entity_descs = await mg_driver.get_entity_descs_for_layer(layer)
print(len(entity_descs))
print(entity_descs[0] if entity_descs else "")

620
{'key': 'detecting_aimbot_usage', 'desc': 'Identifying aimbot usage in video games, such as Minecraft, is essential for maintaining fair play and ensuring the integrity of the gaming experience.'}


In [18]:
# test embed 1 batch
import asyncio
from utils import batched
import litellm

EMBED_MODEL = "bedrock/amazon.titan-embed-text-v2:0"
ENTITY_BATCH_SIZE = 5
MAX_PARALLEL_EMBED = 10

embed_sem = asyncio.Semaphore(MAX_PARALLEL_EMBED)
entity_desc_embeds = []
temp_batch_lim=2
i=0
for batch in batched(entity_descs, ENTITY_BATCH_SIZE):
    if i>=temp_batch_lim: break
    async with embed_sem:
        resp = await litellm.aembedding(model=EMBED_MODEL, input=[e_desc for _,e_desc in batch])
    batch_embed = resp['data']
    # unpack batch to entity_key -> description pairs
    for emb in sorted(batch_embed, key=lambda e: e.index):
        entity_desc_embeds.append({'key':batch[emb.index]['key'],'desc_embed':emb.embedding})
    i+=1


In [19]:
print(entity_desc_embeds)

[{'key': 'detecting_aimbot_usage', 'desc_embed': [-0.05133696645498276, 0.04449614882469177, 0.02701084315776825, -0.013075034134089947, 0.029988937079906464, -0.008253763429820538, 0.06847625225782394, -0.014826788567006588, 0.011733600869774818, 0.005215810611844063, 0.008474212139844894, 0.04465988278388977, 0.0034541920758783817, 0.038700368255376816, -0.026789221912622452, 0.03184179216623306, 0.0009192766738124192, 0.01573028415441513, 0.03486939147114754, 0.027453528717160225, -0.019914373755455017, 0.04764457419514656, 0.015458052046597004, 0.036640871316194534, -0.042073678225278854, -0.011339060962200165, 0.05337328463792801, -0.045472633093595505, -0.012514788657426834, -0.033946167677640915, 0.04773038625717163, -0.0043557146564126015, 0.04229462146759033, 0.010123880580067635, 0.06621948629617691, 0.04512741044163704, -0.004702909383922815, 0.0231929961591959, -0.0006154814036563039, 0.051724109798669815, 0.012471389025449753, 0.0027025947347283363, -0.005192138254642487, 

In [ ]:
# aggregate

# FOR EACH LAYER:
# 1. collect all e_descs=[entity.desc for entity in layer]
# 2. create the set of embeddings e_embeds = [embed(desc) for desc in e_descs]
# 3. Feed e_embeds into GMM to partition the layer into clusters:list

# collect all entity_descs in layer -- temp try with just the json. TODO: memgraph
import json
G0_DESCRIPTION_FILENAME = CWD / "g0_descriptions.json"
description_map = {}
with open(G0_DESCRIPTION_FILENAME, "r") as desc_file:
    data = json.load(desc_file)
    for obj in data:
        description_map.update(obj)
    del data

# create embeddings
# EMBED_BATCH_SIZE = 20


# for batch in batched(description_map.items(), EMBED_BATCH_SIZE):
    # .. use litellm to get embeddings

# example batch embedding list
batched_embeds = [ [1,2,3,4,5.0], [4.6,2.3,9.1,3.4,1,5.6,2.3,5,2.3,1], [ 6.1,1,6,3,1,3,5,3,1,7,7,7,3]]

from sklearn.mixture import GaussianMixture

from typing import NamedTuple
from dataclasses import dataclass

@dataclass
class AggNode:
    name:str
    desc:str
    child_entities:list[str]

def gen_aggregate_entity(cluster:Cluster) -> AggNode:
    """ aggregate generation function (F), creates parent node for cluster"""
# - get all relations between entities in this cluster in this layer : Question : isnt 'in this layer' implied?
# - ask an LLM to generate a name and description given these relations : Question : what exactly do we feed the LLMs
# -> OUTPUT = (new_parent_name, new_parent_description)
    return AggNode(name=agg_name, desc=agg_desc)

def gen_aggregate_rel(cj:Cluster, ck:Cluster):
    """ F(rel)"""
    ...
def concat_rels(relations) -> str:
    ...

new_aggregates = []
for cluster in clusters:
    new_agg_node:AggNode = gen_aggregate(cluster)
    new_agg_node.child_entities = cluster.entities
    new_aggregates.append(new_agg_node)
    # insert the agg parent into the graph, and
    # link each entity in the cluster to this new parent

TAU = ...
# inter-cluster connection:
for cj in new_aggregates:
    for ck in new_aggregates:
        if cj == ck: continue

        # collect all relations between entities in cj and entities in ck
        rel_cj_ck = ... # memgraph cypher
        conn_strength = len(rel_cj_ck)
        
        if conn_strength > TAU:
            agg_rel = gen_aggregate_rel(cj,ck)
        else:
            agg_rel = concat_rels(rel_cj_ck)

        # create relation in memgraph

from utils.mg_driver import get_entity_descs_for_layer
import litellm

gmm = GaussianMixture(...)
async def process_layer(layer:int):
    """Recursive aggregation from G0 -> LeanRAG KG"""
    # collect all entity descriptions from the layer
    entity_descs = await get_entity_descs_for_layer(layer)
    entity_desc_embeds = [] 
    # create embeddings for entity descriptions
    for batch in batched(entity_descs.items()):
        batch_embeds = await litellm.embedding(input=[e_desc for _,e_desc in batch])
        entity_desc_embeds.append(batch_embeds)

    # parititon with GMM        
    partitions = gmm.fit_predict(entity_desc_embeds)
    for cluster in partitions:
        # fetch entities that are part of this cluster
        ...
